Visualise model output from CF registry data vs baseline ppFEV1

In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import cfr.cfr_viz_helpers as vh

In [ ]:
# Load AC with inferred from 2023 data, 2nd day = 2019 data

df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# EXCEL
# 2 entries means there is a 2019 entry for every 2023 entry
# df_res = bd.load_meas_from_excel(
#     "infer_AR_using_19_23_data_2entries_fev1_10122025",
#     # "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
#     study_folder="CFR",
#     str_cols_to_arrays=["Airway resistance (%)"],
# )
# df_res = df_res.drop(columns=["Healthy FEV1 (L)"])

# CSV
# df_res = (
#     bd.load_meas_from_excel(
#         # "infer_AR_using_two_days_model_19_23_data_2entries_fev1_10122025",
#         "infer_AR_using_two_days_model_19_23_data_2entries_fev1_fef2575_10122025",
#         study_folder="CFR",
#         str_cols_to_arrays=["Airway resistance (%)"],
#         use_csv=True,
#         date_cols=["Day"],
#         bypass_sanity_checks=True,
#     )
#     .drop(columns=["Healthy FEV1 (L)"])
#     .rename(columns={"Day": "Date Recorded"})
# )

# Merging
df = df_res.merge(df_meas, on=["ID", "Date Recorded"])

In [3]:
# Load AC from 2019 data with 2nd day = best FEV1 (no FEF2575)
# df = bd.load_meas_from_excel("AR_19_data_with_best_FEV1", study_folder="CFR", str_cols_to_arrays=["Airway resistance (%)"])
df = bd.load_meas_from_excel(
    "infer_all_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(FEV1|pred_FEV1)",
        "P(HFEV1|FEF2575, bFEV1)"
    ],
)
print(f"Shape: {df.shape}")

Shape: (2037, 21)


In [4]:
# Process

# Keep only values from 2023
# df23 = df[df["Date Recorded"] == datetime.date(2023, 1, 1)]

df23 = df

AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df23[AC.name] = df23[AR.name].apply(lambda arr: arr[::-1])
df23["ecFEV1 % Predicted (clipped)"] = df23["ecFEV1 % Predicted"].clip(upper=100)
df23["P(ppFEV1|AC)"] = vh.calc_P_ppFEV1_given_AC(df23, AC)
df["P(ppFEV1|AC) ratioed"] = vh.calc_P_ppFEV1_given_AC(df, AC, corr=True)

# Airway conductance

In [36]:
## FILL ##
ratioed = False
prctile = 10

df_to_plot, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)
# df_to_plot = df23[df23["P(ppFEV1|AC)"] <= t]

# title = f"Dumbell plot for CF Registry 2023, 2019 2nd day, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, 2019 2nd day, FEV1 & FEF25-75 (2entries), {t*100:.2f}% conf. disagreeing"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1 (no FEF25-75)"
title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 & FEF2575 (2entries)"

ac_col = AC.name

fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

ppfev1_row = "ecFEV1 % Predicted (clipped)"
# ppfev1_row = "ecFEV1 % Predicted"

df_to_plot, _, _ = vh.get_dumbell_plot_data(
    df_to_plot, AC.name, AC, ppfev1_row=ppfev1_row
)

# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)}) clipped"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
vh.plot_dumbell_for_df(
    fig, df_mild, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 3
)
vh.plot_dumbell_for_df(
    fig, df_moderate, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 2
)
vh.plot_dumbell_for_df(
    fig, df_severe, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 1
)


fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    # f"{dh.get_path_to_main()}PlotsCFR/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf"
)
# fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 169
# Moderate: 33
# Severe: 2
35.60473898878335


# Non saturatin ppFEV1

In [10]:
ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
df['mean FEV1'] = df.apply(lambda row: ecFEV1.get_mean(row['P(FEV1|pred_FEV1)']), axis=1)

df['model ppFEV1'] = df['FEV1'] / df['mean FEV1'] * 100

In [9]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted', 'idx FEV1',
       'idx FEF2575%FEV1', 'idx best FEV1', 'P(HFEV1|FEF2575, bFEV1)',
       'P(FEV1|pred_FEV1)', 'P(FEV1_obs|pred_FEV1)', 'Airway resistance (%)',
       'Airway conductance (%)', 'ecFEV1 % Predicted (clipped)',
       'P(ppFEV1|AC)', 'P(ppFEV1|AC) ratioed', 'mean FEV1', 'model ppFEV1'],
      dtype='object')

In [17]:
mean = df["model ppFEV1"].apply(ecFEV1.get_std)

In [25]:
def get_dumbell_plot_data_model_ppfev1(df, ac_row, AC, ppfev1_row="ecFEV1 % Predicted"):
    # Avoid modifying the original dataframe
    df_res = df.copy()

    # Get model ppFEV1
    mean = df_res[ac_row]

    df_res[f"{ac_row} mean"] = mean
    df_res[f"{ac_row} low"] = 0.999*mean
    df_res[f"{ac_row} high"] = 1.0001*mean

    # Get Sorted IDs
    ids_sorted = df_res.sort_values(ppfev1_row, ascending=False)["ID"].values
    # Sort by diff
    df_res["Healhtier diff"] = df_res[ac_row] - df_res[ppfev1_row]
    ids_sorted = df_res.sort_values("Healhtier diff")["ID"].values

    # Prepare for Plotting
    # Map 'low' and 'high' to the same name ('dist') to group them using melt
    plot_cols = {
        f"{ac_row} low": f"{ac_row} dist",
        f"{ac_row} high": f"{ac_row} dist",
    }

    df_melted = (
        df_res.rename(columns=plot_cols)
        .melt(
            id_vars=["ID"],
            value_vars=[ppfev1_row, f"{ac_row} dist", f"{ac_row} mean"],
            var_name="measure",
            value_name="value",
        )
        .set_index("ID")
        .loc[ids_sorted]
        .reset_index()
    )

    return df_melted, df_res, ids_sorted

In [28]:
import numpy as np
import plotly.graph_objects as go

import src.models.helpers as mh

In [31]:
def plot_dumbell_for_df_model_ppfev1(fig, df, measures, col):
    ac_dist = measures[0]  # sigma up, sigma down
    ac_mean = measures[1]
    baseline = measures[2]

    mask = df["measure"] == ac_dist
    # for id in df[mask]["ID"].unique():
    #     mask_id = df["ID"] == id
    #     # Add mask
    #     mask_final = mask_id & mask
    #     fig.add_trace(
    #         go.Scatter(
    #             x=df[mask_final]["value"],
    #             y=df[mask_final]["ID"],
    #             mode="lines",
    #             name="ecFEV1 % healthy FEV1 * f(FEF25-75)",
    #             marker=dict(color="red"),
    #             line=dict(width=3),
    #             showlegend=(True if id == df["ID"].unique()[0] else False),
    #         ),
    #         row=1,
    #         col=col,
    #     )
    # Add ac mean marker
    mask = df["measure"] == ac_mean
    fig.add_trace(
        go.Scatter(
            x=df[mask]["value"],
            y=df[mask]["ID"],
            mode="markers",
            marker=dict(color="red", size=4),
            # showlegend=(True if id == df["ID"].unique()[0] else False),
        ),
        row=1,
        col=col,
    )

    mask = df["measure"] == baseline
    ecfev1_prct_pred = df[mask]["value"]
    # Where above 100, set to 100
    # ecfev1_prct_pred = np.clip(ecfev1_prct_pred, 0, 100)
    fig.add_trace(
        go.Scatter(
            x=ecfev1_prct_pred,
            y=df[mask]["ID"],
            mode="markers",
            name="ecFEV1%Predicted",
            marker=dict(size=4, color="blue"),
        ),
        row=1,
        col=col,
    )

In [24]:
df_to_plot

,ID,measure,value
0,B158415,ecFEV1 % Predicted,91.704893
1,B158415,model ppFEV1 dist,75.628717
2,B158415,model ppFEV1 dist,75.628717
3,B158415,model ppFEV1 mean,75.628717
4,B160593,ecFEV1 % Predicted,91.479101
...,...,...,...
811,B162211,model ppFEV1 mean,58.952421
812,B169339,ecFEV1 % Predicted,64.326504
813,B169339,model ppFEV1 dist,66.905680
814,B169339,model ppFEV1 dist,66.905680


In [33]:
## FILL ##
ratioed = True
prctile = 100

df_to_plot, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)
col = "P(FEV1_obs|pred_FEV1)"
t = df[col].quantile(prctile / 100)
df_to_plot = df[df[col] <= t]

# title = f"Dumbell plot for CF Registry 2023, 2019 2nd day, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, 2019 2nd day, FEV1 & FEF25-75 (2entries), {t*100:.2f}% conf. disagreeing"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1 (no FEF25-75)"
title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(FEV1_obs|pred_FEV1) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 & FEF2575 (2entries)"

ac_col = AC.name
ac_col = "model ppFEV1"

fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

ppfev1_row = "ecFEV1 % Predicted"

df_to_plot, _, _ = get_dumbell_plot_data_model_ppfev1(
    df_to_plot, ac_col, ecFEV1, ppfev1_row=ppfev1_row
)

# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)}) clipped"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
plot_dumbell_for_df_model_ppfev1(
    fig, df_mild, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 3
)
plot_dumbell_for_df_model_ppfev1(
    fig, df_moderate, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 2
)
plot_dumbell_for_df_model_ppfev1(
    fig, df_severe, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 1
)


fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    # f"{dh.get_path_to_main()}PlotsCFR/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf"
)
# fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 890
# Moderate: 793
# Severe: 354
13.3130788065074
